# EEG outlier review and cleaning

This notebook is for the **Siena Scalp EEG Database** used by `final_project`. It never modifies `data/raw`.

EEG requires a different policy from ordinary tabular data. Large amplitudes can be seizure physiology, eye or muscle activity, movement, baseline drift, or electrode failure. This notebook therefore audits every available recording and defines one reproducible, window-level cleaning function for the team's models. It never modifies or replaces the raw EDF files.

- global statistical amplitude outliers are **review flags**, not automatic deletions;
- auxiliary EKG, pulse-oximetry, heart-rate, marker, and unnamed signals are excluded from the EEG dataset;
- channel validity is decided independently in short analysis windows after detrending and robust common-median referencing;
- flat, materially clipped, or non-finite channels are rejected for that window only;
- detrending and robust re-referencing happen before amplitude capping;
- values still above +500 µV after cleaning are capped at +500 µV, and values still below -500 µV are capped at -500 µV.

## Phase 1 — Locate sources and protect raw data

The expected recording list and preserved raw-data audit are read as source material. We also count locally available EDF files. A missing EDF source is reported explicitly rather than silently treating audit statistics as cleaned signal samples.

In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from scipy import signal

pd.set_option("display.max_columns", None)

def find_project_dir(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw" / "RECORDS").exists():
            return candidate
        nested = candidate / "26-the-optimizers-analysis" / "final_project"
        if (nested / "data" / "raw" / "RECORDS").exists():
            return nested
    raise FileNotFoundError("Could not locate final_project/data/raw/RECORDS")

PROJECT_DIR = find_project_dir(Path.cwd().resolve())
RAW_DIR = PROJECT_DIR / "data" / "raw"
INTERIM_DIR = PROJECT_DIR / "data" / "interim"
AUDIT_DIR = PROJECT_DIR / "results" / "raw_data_audit"
METRICS_PATH = AUDIT_DIR / "sampled_channel_metrics.csv"
INVENTORY_PATH = AUDIT_DIR / "edf_file_inventory.csv"

expected_recordings = [line.strip() for line in (RAW_DIR / "RECORDS").read_text().splitlines() if line.strip()]
local_edfs = sorted(RAW_DIR.rglob("*.edf"))
inventory = pd.read_csv(INVENTORY_PATH)
metrics = pd.read_csv(METRICS_PATH)

print(f"Project: {PROJECT_DIR}")
print(f"Expected EDF recordings: {len(expected_recordings)}")
print(f"Locally materialized EDF recordings: {len(local_edfs)}")
print(f"Preserved channel-audit rows: {len(metrics):,}")
print(f"Header-derived source size: {inventory['file_size_mb'].sum() / 1024:.2f} GB")

## Phase 2 — Keep scalp EEG and normalize channel names

Only labels beginning with `EEG` are scalp-EEG channels. Label case varies in the source (`Fp2` versus `FP2`, `Cz` versus `CZ`), so labels are normalized to uppercase electrode codes for reliable pooling. Numeric values are validated as microvolts.

In [ ]:
is_eeg = metrics["channel"].str.match(r"^EEG\s+", case=False, na=False)
eeg = metrics.loc[is_eeg].copy()
auxiliary = metrics.loc[~is_eeg].copy()

def normalize_eeg_label(label: str) -> str:
    electrode = re.sub(r"^EEG\s+", "", str(label), flags=re.IGNORECASE).strip()
    return electrode.upper()

eeg = eeg.rename(columns={"channel": "channel_original"})
eeg.insert(2, "channel_normalized", eeg["channel_original"].map(normalize_eeg_label))
numeric_columns = ["samples_sampled", "mean", "std", "rms", "p005", "p995", "max_abs", "flat_fraction", "clipped_fraction"]
nonfinite_measurement = ~np.isfinite(eeg[numeric_columns].astype(float)).all(axis=1)
wrong_unit = eeg["unit"].ne("uV")

print(f"Scalp-EEG channel/recording pairs retained for QC: {len(eeg):,}")
print(f"Auxiliary channel/recording pairs excluded: {len(auxiliary):,}")
print(f"Normalized electrode names: {eeg['channel_normalized'].nunique()}")
print(f"Non-finite metric rows: {nonfinite_measurement.sum()}")
print(f"Non-microvolt EEG rows: {wrong_unit.sum()}")

## Phase 3 — Find every recording outside ±500 µV

Three complementary review screens are applied to the preserved 40-second-per-channel audit:

1. A global 1.5×IQR high-RMS flag reproduces the original audit's broad amplitude review.
2. A modified z-score on `log1p(RMS)` compares each channel with other channels in the same recording; absolute scores above 3.5 receive focused review.
3. Near-flat, materially clipped, non-finite, or wrong-unit audit rows are invalid source rows and may be excluded automatically.

The preserved audit provides a quick sampled flag at ±500 µV. When EDF files are locally available, the second half of this phase reads **every EEG sample** in bounded-size chunks and builds an exact recording-level table containing separate counts above +500 µV and below -500 µV.

In [ ]:
q1, q3 = eeg["rms"].quantile([0.25, 0.75])
iqr = q3 - q1
rms_lower = q1 - 1.5 * iqr
rms_upper = q3 + 1.5 * iqr

log_rms = np.log1p(eeg["rms"])
file_median = log_rms.groupby(eeg["file"]).transform("median")
file_mad = log_rms.groupby(eeg["file"]).transform(lambda values: np.median(np.abs(values - np.median(values))))
modified_z = pd.Series(np.where(file_mad > 0, 0.6745 * (log_rms - file_median) / file_mad, 0.0), index=eeg.index)

eeg["global_high_rms_review"] = eeg["rms"] > rms_upper
eeg["within_recording_rms_review"] = modified_z.abs() > 3.5
eeg["within_recording_modified_z"] = modified_z.round(4)
AMPLITUDE_CAP_UV = 500.0
eeg["sampled_outside_500uv_review"] = eeg["max_abs"] > AMPLITUDE_CAP_UV
eeg["near_flat_invalid"] = eeg["flat_fraction"] >= 0.95
eeg["material_clipping_invalid"] = eeg["clipped_fraction"] >= 0.01
eeg["nonfinite_metrics_invalid"] = nonfinite_measurement
eeg["wrong_unit_invalid"] = wrong_unit

invalid_columns = ["near_flat_invalid", "material_clipping_invalid", "nonfinite_metrics_invalid", "wrong_unit_invalid"]
review_columns = ["global_high_rms_review", "within_recording_rms_review", "sampled_outside_500uv_review"]
eeg["exclude_entire_channel_recording"] = eeg[invalid_columns].any(axis=1)
eeg["requires_manual_review"] = eeg[review_columns].any(axis=1)

print(f"Global high-RMS review bound: > {rms_upper:.1f} µV")
print(f"Global high-RMS review flags: {eeg['global_high_rms_review'].sum()}")
print(f"Within-recording robust review flags: {eeg['within_recording_rms_review'].sum()}")
print(f"Sampled outside ±500 µV review flags: {eeg['sampled_outside_500uv_review'].sum()}")
print(f"Invalid channel/recording rows requiring automatic exclusion: {eeg['exclude_entire_channel_recording'].sum()}")

def unit_to_microvolts(unit: str) -> float:
    normalized = str(unit).strip().lower().replace("µ", "u").replace("μ", "u")
    factors = {"uv": 1.0, "mv": 1_000.0, "v": 1_000_000.0}
    if normalized not in factors:
        raise ValueError(f"Unsupported EEG physical dimension: {unit!r}")
    return factors[normalized]

def _read_edf_layout(edf_path: Path) -> dict:
    """Read the fixed-width EDF header fields needed for a direct, dependency-free scan."""
    with edf_path.open("rb") as handle:
        fixed = handle.read(256)
        if len(fixed) != 256:
            raise ValueError(f"Incomplete EDF header: {edf_path}")
        header_bytes = int(fixed[184:192].decode("ascii").strip())
        n_records = int(fixed[236:244].decode("ascii").strip())
        n_signals = int(fixed[252:256].decode("ascii").strip())
        signal_header = handle.read(n_signals * 256)
    if len(signal_header) != n_signals * 256:
        raise ValueError(f"Incomplete EDF signal header: {edf_path}")

    cursor = 0
    def read_field(width: int) -> list[str]:
        nonlocal cursor
        values = [
            signal_header[cursor + index * width:cursor + (index + 1) * width]
            .decode("ascii", errors="replace").strip()
            for index in range(n_signals)
        ]
        cursor += width * n_signals
        return values

    labels = read_field(16)
    read_field(80)  # transducer
    units = read_field(8)
    physical_min = np.asarray(read_field(8), dtype=float)
    physical_max = np.asarray(read_field(8), dtype=float)
    digital_min = np.asarray(read_field(8), dtype=float)
    digital_max = np.asarray(read_field(8), dtype=float)
    read_field(80)  # prefilter
    samples_per_record = np.asarray(read_field(8), dtype=int)
    read_field(32)  # reserved

    samples_in_record = int(samples_per_record.sum())
    if n_records < 0:
        data_bytes = edf_path.stat().st_size - header_bytes
        n_records = data_bytes // (2 * samples_in_record)
    expected_bytes = header_bytes + n_records * samples_in_record * 2
    if expected_bytes > edf_path.stat().st_size:
        raise ValueError(f"EDF data are shorter than the header declares: {edf_path}")
    if np.any(digital_max == digital_min):
        raise ValueError(f"EDF channel has equal digital minimum and maximum: {edf_path}")
    return {
        "header_bytes": header_bytes,
        "n_records": n_records,
        "n_signals": n_signals,
        "labels": labels,
        "units": units,
        "physical_min": physical_min,
        "physical_max": physical_max,
        "digital_min": digital_min,
        "digital_max": digital_max,
        "samples_per_record": samples_per_record,
        "samples_in_record": samples_in_record,
    }

def scan_recording_amplitude_bounds(edf_path: Path, chunk_records: int = 256) -> list[dict]:
    """Count every scalp-EEG sample outside ±500 µV in one sequential EDF pass."""
    layout = _read_edf_layout(edf_path)
    offsets = np.concatenate(([0], np.cumsum(layout["samples_per_record"])))
    eeg_indices = [
        index for index, label in enumerate(layout["labels"])
        if re.match(r"^EEG\s+", label, re.IGNORECASE)
    ]
    rows = [{
        "file": edf_path.relative_to(RAW_DIR).as_posix(),
        "channel_original": layout["labels"][index],
        "channel_normalized": normalize_eeg_label(layout["labels"][index]),
        "samples_above_500uv": 0,
        "samples_below_minus_500uv": 0,
        "samples_to_cap": 0,
        "nonfinite_samples": 0,
        "observed_min_uv": np.inf,
        "observed_max_uv": -np.inf,
    } for index in eeg_indices]

    digital_records = np.memmap(
        edf_path,
        dtype="<i2",
        mode="r",
        offset=layout["header_bytes"],
        shape=(layout["n_records"], layout["samples_in_record"]),
    )
    for record_start in range(0, layout["n_records"], chunk_records):
        record_stop = min(record_start + chunk_records, layout["n_records"])
        record_block = digital_records[record_start:record_stop]
        for row, channel_index in zip(rows, eeg_indices, strict=True):
            digital = record_block[:, offsets[channel_index]:offsets[channel_index + 1]].reshape(-1)
            scale = (
                (layout["physical_max"][channel_index] - layout["physical_min"][channel_index])
                / (layout["digital_max"][channel_index] - layout["digital_min"][channel_index])
            )
            physical = (
                (digital.astype(np.float64) - layout["digital_min"][channel_index]) * scale
                + layout["physical_min"][channel_index]
            )
            values_uv = physical * unit_to_microvolts(layout["units"][channel_index])
            finite = np.isfinite(values_uv)
            row["nonfinite_samples"] += int((~finite).sum())
            if finite.any():
                finite_values = values_uv[finite]
                row["samples_above_500uv"] += int((finite_values > AMPLITUDE_CAP_UV).sum())
                row["samples_below_minus_500uv"] += int((finite_values < -AMPLITUDE_CAP_UV).sum())
                row["observed_min_uv"] = min(row["observed_min_uv"], float(finite_values.min()))
                row["observed_max_uv"] = max(row["observed_max_uv"], float(finite_values.max()))
    del digital_records

    for row in rows:
        row["samples_to_cap"] = row["samples_above_500uv"] + row["samples_below_minus_500uv"]
        if not np.isfinite(row["observed_min_uv"]): row["observed_min_uv"] = np.nan
        if not np.isfinite(row["observed_max_uv"]): row["observed_max_uv"] = np.nan
    return rows

amplitude_columns = [
    "file", "channel_original", "channel_normalized", "samples_above_500uv",
    "samples_below_minus_500uv", "samples_to_cap", "nonfinite_samples",
    "observed_min_uv", "observed_max_uv",
]
amplitude_rows = [row for edf_path in local_edfs for row in scan_recording_amplitude_bounds(edf_path)]
channel_amplitude_audit = pd.DataFrame(amplitude_rows, columns=amplitude_columns)
if channel_amplitude_audit.empty:
    recordings_outside_bounds = pd.DataFrame(columns=[
        "file", "samples_above_500uv", "samples_below_minus_500uv", "samples_to_cap"
    ])
else:
    recordings_outside_bounds = (
        channel_amplitude_audit.groupby("file", as_index=False)[
            ["samples_above_500uv", "samples_below_minus_500uv", "samples_to_cap"]
        ].sum().query("samples_to_cap > 0").sort_values("samples_to_cap", ascending=False)
    )

print(f"Recordings with at least one sample outside ±500 µV: {len(recordings_outside_bounds)}")
display(recordings_outside_bounds)

## Phase 4 — Make the cleaning decision

Statistical amplitude flags remain eligible for window-level cleaning because seizure EEG is expected to contain unusual amplitudes. Only structurally invalid channel/recording rows are excluded globally. Every retained channel is still re-evaluated inside every short window.

In [ ]:
def review_reason(row: pd.Series) -> str:
    reasons = []
    if row["global_high_rms_review"]: reasons.append("global high RMS")
    if row["within_recording_rms_review"]: reasons.append("within-recording robust RMS outlier")
    if row["sampled_outside_500uv_review"]: reasons.append("sampled absolute amplitude >500 uV")
    if row["near_flat_invalid"]: reasons.append("near-flat")
    if row["material_clipping_invalid"]: reasons.append("material clipping")
    if row["nonfinite_metrics_invalid"]: reasons.append("non-finite audit metrics")
    if row["wrong_unit_invalid"]: reasons.append("unexpected unit")
    return "; ".join(reasons)

eeg["review_reason"] = eeg.apply(review_reason, axis=1)
eeg["cleaning_action"] = np.where(
    eeg["exclude_entire_channel_recording"],
    "exclude channel from recording",
    "retain; apply window-level QC",
)

clean_manifest = eeg.loc[~eeg["exclude_entire_channel_recording"]].copy()
outlier_review = eeg.loc[eeg["requires_manual_review"] | eeg["exclude_entire_channel_recording"]].copy()

print(f"EEG channel/recording pairs in clean manifest: {len(clean_manifest):,}")
print(f"Pairs excluded globally: {eeg['exclude_entire_channel_recording'].sum()}")
print(f"Pairs retained but prioritized for manual review: {eeg['requires_manual_review'].sum()}")
display(outlier_review[["file", "channel_original", "channel_normalized", "rms", "max_abs", "review_reason", "cleaning_action"]].head(15))

## Phase 5 — Define the reusable window-level cleaner

The function below implements the project method on an array shaped `(channels, samples)`:

1. record raw excursions and hardware saturation without changing them;
2. reject non-finite, nearly flat (`detrended SD < 0.5 µV`), or materially hardware-clipped (`>1%`) channels;
3. linearly detrend the remaining channels to remove baseline drift;
4. subtract the across-channel median at each sample as a robust reference;
5. apply a 60 Hz notch filter where the sample rate permits it;
6. cap only the remaining cleaned values to `[-500, 500]` µV.

Rejected channels are represented by the returned boolean mask; they are not interpolated. The QC table distinguishes raw excursions from values capped after preprocessing. `read_and_clean_edf_window(...)` loads a requested EDF interval, applies exactly this cleaner, and returns the retained channel names for model use. Both synthetic and real-data smoke tests are included.

In [ ]:
MIN_STD_UV = 0.5
MAX_HARDWARE_CLIPPED_FRACTION = 0.01
MIN_USABLE_CHANNELS = 10
NOTCH_FREQUENCY_HZ = 60.0
CLEANING_VERSION = "detrend-reference-qc-cap-v1"

def clean_eeg_window(
    data_uv: np.ndarray,
    sample_rate_hz: float,
    physical_min_uv: np.ndarray,
    physical_max_uv: np.ndarray,
    min_usable_channels: int = MIN_USABLE_CHANNELS,
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    """Clean one `(channels, samples)` EEG window without modifying its input."""
    data_uv = np.asarray(data_uv, dtype=float)
    if data_uv.ndim != 2 or data_uv.shape[1] < 2:
        raise ValueError("data_uv must have shape (channels, samples) with at least two samples")
    if not np.isfinite(sample_rate_hz) or sample_rate_hz <= 0:
        raise ValueError("sample_rate_hz must be positive and finite")
    physical_min_uv = np.asarray(physical_min_uv, dtype=float)
    physical_max_uv = np.asarray(physical_max_uv, dtype=float)
    expected_limits_shape = (data_uv.shape[0],)
    if physical_min_uv.shape != expected_limits_shape or physical_max_uv.shape != expected_limits_shape:
        raise ValueError("Physical limits must contain one value per channel")

    finite_input = np.isfinite(data_uv).all(axis=1)
    raw_above_cap = np.sum(data_uv > AMPLITUDE_CAP_UV, axis=1)
    raw_below_cap = np.sum(data_uv < -AMPLITUDE_CAP_UV, axis=1)
    tolerance = np.maximum((physical_max_uv - physical_min_uv) * 0.001, 1e-6)
    hardware_clipped_fraction = np.mean(
        (data_uv <= physical_min_uv[:, None] + tolerance[:, None])
        | (data_uv >= physical_max_uv[:, None] - tolerance[:, None]),
        axis=1,
    )

    safe_input = np.where(np.isfinite(data_uv), data_uv, 0.0)
    detrended = signal.detrend(safe_input, axis=1, type="linear")
    detrended_std = np.std(detrended, axis=1)
    preliminary_usable = (
        finite_input
        & (detrended_std >= MIN_STD_UV)
        & (hardware_clipped_fraction <= MAX_HARDWARE_CLIPPED_FRACTION)
    )
    if int(preliminary_usable.sum()) < min_usable_channels:
        raise ValueError(
            f"Only {int(preliminary_usable.sum())} channels passed pre-reference QC; "
            f"{min_usable_channels} required"
        )

    reference = np.median(detrended[preliminary_usable], axis=0, keepdims=True)
    referenced = detrended - reference
    post_reference_std = np.std(referenced, axis=1)
    post_reference_max_abs = np.max(np.abs(referenced), axis=1)
    usable = preliminary_usable & np.isfinite(referenced).all(axis=1)
    cleaned_before_cap = referenced[usable]

    if sample_rate_hz > 2 * NOTCH_FREQUENCY_HZ + 2:
        notch_b, notch_a = signal.iirnotch(NOTCH_FREQUENCY_HZ, 30.0, fs=sample_rate_hz)
        cleaned_before_cap = signal.filtfilt(notch_b, notch_a, cleaned_before_cap, axis=1)

    final_above_cap = np.sum(cleaned_before_cap > AMPLITUDE_CAP_UV, axis=1)
    final_below_cap = np.sum(cleaned_before_cap < -AMPLITUDE_CAP_UV, axis=1)
    cleaned = np.clip(cleaned_before_cap, -AMPLITUDE_CAP_UV, AMPLITUDE_CAP_UV)

    qc = pd.DataFrame({
        "finite_input": finite_input,
        "raw_samples_above_500uv": raw_above_cap,
        "raw_samples_below_minus_500uv": raw_below_cap,
        "hardware_clipped_fraction": hardware_clipped_fraction,
        "detrended_std_uv": detrended_std,
        "post_reference_std_uv": post_reference_std,
        "post_reference_max_abs_uv": post_reference_max_abs,
        "preliminary_usable": preliminary_usable,
        "usable": usable,
        "final_samples_above_500uv_capped": 0,
        "final_samples_below_minus_500uv_capped": 0,
        "final_samples_capped": 0,
    })
    qc.loc[usable, "final_samples_above_500uv_capped"] = final_above_cap
    qc.loc[usable, "final_samples_below_minus_500uv_capped"] = final_below_cap
    qc.loc[usable, "final_samples_capped"] = final_above_cap + final_below_cap
    return cleaned.astype(np.float32), usable, qc

def read_and_clean_edf_window(
    edf_path: str | Path,
    start_seconds: float,
    duration_seconds: float,
    min_usable_channels: int = MIN_USABLE_CHANNELS,
) -> tuple[np.ndarray, float, list[str], pd.DataFrame]:
    """Read scalp EEG from one EDF interval and apply `clean_eeg_window`."""
    import pyedflib

    edf_path = Path(edf_path)
    if start_seconds < 0 or duration_seconds <= 0:
        raise ValueError("Requested EDF interval has invalid bounds")
    reader = pyedflib.EdfReader(str(edf_path))
    try:
        labels = [str(label).strip() for label in reader.getSignalLabels()]
        eeg_indices = [index for index, label in enumerate(labels) if re.match(r"^EEG\s+", label, re.IGNORECASE)]
        if len(eeg_indices) < min_usable_channels:
            raise ValueError(f"Only {len(eeg_indices)} scalp-EEG channels found in {edf_path.name}")
        rates = np.asarray([reader.getSampleFrequency(index) for index in eeg_indices], dtype=float)
        unique_rates, counts = np.unique(np.round(rates, 6), return_counts=True)
        sample_rate_hz = float(unique_rates[np.argmax(counts)])
        eeg_indices = [
            index for index in eeg_indices
            if np.isclose(reader.getSampleFrequency(index), sample_rate_hz, atol=1e-6, rtol=0)
        ]
        start_sample = int(round(start_seconds * sample_rate_hz))
        sample_count = int(round(duration_seconds * sample_rate_hz))
        if start_sample + sample_count > int(reader.getNSamples()[eeg_indices[0]]):
            raise ValueError(f"Requested interval extends beyond {edf_path.name}")

        factors = np.asarray([unit_to_microvolts(reader.getPhysicalDimension(index)) for index in eeg_indices])
        data_uv = np.vstack([
            reader.readSignal(index, start=start_sample, n=sample_count) * factor
            for index, factor in zip(eeg_indices, factors, strict=True)
        ])
        physical_min_uv = np.asarray([reader.getPhysicalMinimum(index) for index in eeg_indices]) * factors
        physical_max_uv = np.asarray([reader.getPhysicalMaximum(index) for index in eeg_indices]) * factors
        selected_labels = [labels[index] for index in eeg_indices]
    finally:
        reader.close()

    cleaned, usable, qc = clean_eeg_window(
        data_uv,
        sample_rate_hz,
        physical_min_uv,
        physical_max_uv,
        min_usable_channels=min_usable_channels,
    )
    qc.insert(0, "channel_original", selected_labels)
    qc.insert(1, "channel_normalized", [normalize_eeg_label(label) for label in selected_labels])
    retained_labels = [label for label, keep in zip(selected_labels, usable, strict=True) if keep]
    return cleaned, sample_rate_hz, retained_labels, qc

rng = np.random.default_rng(42)
synthetic = rng.normal(0, 20, size=(12, 5 * 512))
synthetic[0] = 0
synthetic[1, 100] = 20_000
synthetic[2, 200] = -20_000
synthetic_clean, synthetic_mask, synthetic_qc = clean_eeg_window(
    synthetic, 512.0, np.full(12, -4096.0), np.full(12, 4096.0)
)
assert synthetic_clean.shape == (11, 5 * 512)
assert not synthetic_mask[0] and synthetic_mask[1] and synthetic_mask[2]
assert synthetic_qc.loc[1, "raw_samples_above_500uv"] == 1
assert synthetic_qc.loc[2, "raw_samples_below_minus_500uv"] == 1
assert np.isfinite(synthetic_clean).all()
assert synthetic_clean.max() <= AMPLITUDE_CAP_UV and synthetic_clean.min() >= -AMPLITUDE_CAP_UV
print("Synthetic test passed: detrend/re-reference precede the final ±500 µV cap.")

if local_edfs:
    real_clean, real_rate, real_labels, real_qc = read_and_clean_edf_window(local_edfs[0], 0.0, 5.0)
    assert real_clean.shape[0] == len(real_labels) and real_clean.shape[1] == int(round(5.0 * real_rate))
    assert np.isfinite(real_clean).all()
    assert real_clean.max() <= AMPLITUDE_CAP_UV and real_clean.min() >= -AMPLITUDE_CAP_UV
    print(f"Real-data smoke test passed: {local_edfs[0].name}, {len(real_labels)} retained channels.")

## Phase 6 — Write team-reproducible artifacts

The notebook writes small manifests and a versioned cleaning configuration to `data/interim`; these files can be regenerated from each teammate's local EDF copy. It deliberately does not create replacement EDF files. Models should load the raw interval they need through `read_and_clean_edf_window(...)`, or use the same function while generating their local feature cache.

In [ ]:
MANIFEST_PATH = INTERIM_DIR / "eeg_clean_channel_manifest.csv"
OUTLIER_PATH = INTERIM_DIR / "eeg_outlier_review.csv"
SUMMARY_PATH = INTERIM_DIR / "eeg_cleaning_summary.json"
CLEANING_CONFIG_PATH = INTERIM_DIR / "eeg_cleaning_config.json"
CHANNEL_AMPLITUDE_AUDIT_PATH = INTERIM_DIR / "eeg_500uv_channel_audit.csv"
RECORDING_AMPLITUDE_AUDIT_PATH = INTERIM_DIR / "eeg_recordings_outside_500uv.csv"

manifest_columns = [
    "patient_id", "file", "channel_original", "channel_normalized", "unit",
    "samples_sampled", "mean", "std", "rms", "p005", "p995", "max_abs",
    "flat_fraction", "clipped_fraction", "global_high_rms_review",
    "within_recording_rms_review", "within_recording_modified_z",
    "sampled_outside_500uv_review", "requires_manual_review", "review_reason",
    "cleaning_action",
]
review_columns_output = manifest_columns + invalid_columns + ["exclude_entire_channel_recording"]

cleaning_config = {
    "cleaning_version": CLEANING_VERSION,
    "input": "raw EDF window; raw files remain unchanged",
    "processing_order": [
        "record raw amplitude and hardware-saturation QC",
        "linear detrend per channel",
        "reject non-finite, flat, and materially hardware-clipped channels",
        "common-median reference using preliminary usable channels",
        "60 Hz notch filter when permitted by sample rate",
        "cap remaining samples to [-500, 500] microvolts",
    ],
    "amplitude_cap_uv": AMPLITUDE_CAP_UV,
    "minimum_detrended_std_uv": MIN_STD_UV,
    "maximum_hardware_clipped_fraction": MAX_HARDWARE_CLIPPED_FRACTION,
    "minimum_usable_channels": MIN_USABLE_CHANNELS,
    "notch_frequency_hz": NOTCH_FREQUENCY_HZ,
}

summary = {
    "dataset": "Siena Scalp EEG Database 1.0.0",
    "cleaning_version": CLEANING_VERSION,
    "expected_edf_recordings": len(expected_recordings),
    "local_edf_recordings": len(local_edfs),
    "source_size_gb": round(float(inventory["file_size_mb"].sum() / 1024), 2),
    "audit_channel_rows": len(metrics),
    "auxiliary_rows_excluded": len(auxiliary),
    "eeg_channel_recording_pairs": len(eeg),
    "globally_excluded_eeg_pairs": int(eeg["exclude_entire_channel_recording"].sum()),
    "clean_manifest_pairs": len(clean_manifest),
    "manual_review_pairs": int(eeg["requires_manual_review"].sum()),
    "global_high_rms_review_bound_uv": round(float(rms_upper), 4),
    "global_high_rms_flags": int(eeg["global_high_rms_review"].sum()),
    "within_recording_robust_rms_flags": int(eeg["within_recording_rms_review"].sum()),
    "amplitude_cap_uv": AMPLITUDE_CAP_UV,
    "sampled_outside_500uv_flags": int(eeg["sampled_outside_500uv_review"].sum()),
    "recordings_scanned_completely": len(local_edfs),
    "recordings_outside_500uv": len(recordings_outside_bounds),
    "samples_above_500uv": int(recordings_outside_bounds["samples_above_500uv"].sum()),
    "samples_below_minus_500uv": int(recordings_outside_bounds["samples_below_minus_500uv"].sum()),
    "full_recording_scan_complete": len(local_edfs) == len(expected_recordings),
    "full_recording_scan_note": (
        "All expected raw EDFs were scanned sample by sample."
        if len(local_edfs) == len(expected_recordings)
        else f"Scanned {len(local_edfs)} of {len(expected_recordings)} expected EDF recordings; download the missing EDFs and rerun for a complete audit."
    ),
}

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
clean_manifest[manifest_columns].to_csv(MANIFEST_PATH, index=False)
outlier_review[review_columns_output].to_csv(OUTLIER_PATH, index=False)
channel_amplitude_audit.to_csv(CHANNEL_AMPLITUDE_AUDIT_PATH, index=False)
recordings_outside_bounds.to_csv(RECORDING_AMPLITUDE_AUDIT_PATH, index=False)
SUMMARY_PATH.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
CLEANING_CONFIG_PATH.write_text(json.dumps(cleaning_config, indent=2) + "\n", encoding="utf-8")

assert pd.read_csv(MANIFEST_PATH).shape[0] == len(clean_manifest)
assert pd.read_csv(OUTLIER_PATH).shape[0] == len(outlier_review)
assert pd.read_csv(CHANNEL_AMPLITUDE_AUDIT_PATH).shape[0] == len(channel_amplitude_audit)
assert pd.read_csv(RECORDING_AMPLITUDE_AUDIT_PATH).shape[0] == len(recordings_outside_bounds)
assert json.loads(CLEANING_CONFIG_PATH.read_text(encoding="utf-8"))["cleaning_version"] == CLEANING_VERSION
print(f"Wrote: {MANIFEST_PATH}")
print(f"Wrote: {OUTLIER_PATH}")
print(f"Wrote: {CHANNEL_AMPLITUDE_AUDIT_PATH}")
print(f"Wrote: {RECORDING_AMPLITUDE_AUDIT_PATH}")
print(f"Wrote: {CLEANING_CONFIG_PATH}")
print(f"Wrote: {SUMMARY_PATH}")
display(pd.Series(summary).to_frame("result"))